# Provider-Level Feature Engineering

This notebook builds a modeling dataset directly from PostgreSQL.

## Goals
- Aggregate provider-level metrics
- Create features:
  - total_services
  - total_beneficiaries
  - avg_charge
  - avg_payment
  - charge_payment_ratio
  - unique_procedures
  - years_active
- Create and export modeling dataset

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f"{x:,.4f}")

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5433')
DB_NAME = os.getenv('DB_NAME', 'medicare_provider_analytics')
DB_USER = os.getenv('DB_USER', 'medicare_user')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'medicare_password')

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


def run_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)


print(f"Connected target: {DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
provider_feature_sql = """
WITH provider_agg AS (
    SELECT
        p.provider_key,
        p.npi,
        p.provider_type,
        SUM(COALESCE(f.tot_srvcs, 0)) AS total_services,
        SUM(COALESCE(f.tot_benes, 0)) AS total_beneficiaries,
        SUM(COALESCE(f.avg_sbmtd_chrg, 0) * COALESCE(f.tot_srvcs, 0)) AS weighted_charge_sum,
        SUM(COALESCE(f.avg_mdcr_pymt_amt, 0) * COALESCE(f.tot_srvcs, 0)) AS weighted_payment_sum,
        COUNT(DISTINCT s.hcpcs_code) AS unique_procedures,
        COUNT(DISTINCT f.data_year) AS years_active
    FROM fact_provider_service f
    JOIN dim_provider p
        ON f.provider_key = p.provider_key
    JOIN dim_service s
        ON f.service_key = s.service_key
    GROUP BY
        p.provider_key,
        p.npi,
        p.provider_type
)
SELECT
    provider_key,
    npi,
    provider_type,
    total_services,
    total_beneficiaries,
    CASE
        WHEN total_services = 0 THEN NULL
        ELSE weighted_charge_sum / total_services
    END AS avg_charge,
    CASE
        WHEN total_services = 0 THEN NULL
        ELSE weighted_payment_sum / total_services
    END AS avg_payment,
    CASE
        WHEN weighted_payment_sum = 0 THEN NULL
        ELSE weighted_charge_sum / weighted_payment_sum
    END AS charge_payment_ratio,
    unique_procedures,
    years_active
FROM provider_agg
ORDER BY provider_key;
"""

df_features = run_sql(provider_feature_sql)

# Basic null handling for modeling dataset stability.
model_numeric_cols = [
    'total_services',
    'total_beneficiaries',
    'avg_charge',
    'avg_payment',
    'charge_payment_ratio',
    'unique_procedures',
    'years_active',
]

for col in model_numeric_cols:
    df_features[col] = pd.to_numeric(df_features[col], errors='coerce')

# Keep a clean modeling dataset with deterministic fills.
df_model = df_features.copy()
df_model[model_numeric_cols] = df_model[model_numeric_cols].replace([np.inf, -np.inf], np.nan)
df_model[model_numeric_cols] = df_model[model_numeric_cols].fillna(0)

display(df_model.head())
print(f"Providers in modeling dataset: {len(df_model):,}")

In [ ]:
summary = pd.DataFrame({
    'feature': model_numeric_cols,
    'min': [df_model[c].min() for c in model_numeric_cols],
    'median': [df_model[c].median() for c in model_numeric_cols],
    'mean': [df_model[c].mean() for c in model_numeric_cols],
    'max': [df_model[c].max() for c in model_numeric_cols],
})

display(summary)

print('Missing values after cleaning:')
display(df_model[model_numeric_cols].isna().sum().to_frame('null_count'))

In [ ]:
output_path = Path.cwd().parent / 'Data' / 'processed' / 'provider_modeling_dataset.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

df_model.to_csv(output_path, index=False)

print(f"Modeling dataset saved to: {output_path}")
print(f"Shape: {df_model.shape[0]:,} rows x {df_model.shape[1]:,} columns")

display(df_model.head(10))